In [ ]:
import yfinance as yf
import pandas as pd

def backtest(symbol, start, end, targetValue, thresBuy, thresSell,
             tradeRatioBuy=1.0, tradeRatioSell=1.0, maxBuys=None):

    # --- Daten laden ---
    tk   = yf.Ticker(symbol)
    hist = tk.history(start=start, end=end)
    # hist.to_csv("~/Schreibtisch/" + symbol + ".csv")
    divs = tk.dividends

    divs = divs[(divs.index >= hist.index[0]) & (divs.index <= hist.index[-1])]
    divs.index = divs.index.tz_localize(None) if divs.index.tzinfo else divs.index
    hist.index = hist.index.tz_localize(None) if hist.index.tzinfo else hist.index

    divsDates = [dat.strftime("%Y-%m-%d") for dat in divs.index]

    # --- Initialer Kauf ---
    priceFirst = hist["Close"].iloc[0]
    shares     = targetValue / priceFirst
    cashFlow   = -targetValue
    cntBuys    = 0
    trades     = []

    trades.append({
        "Datum"    : hist.index[0].date(),
        "Typ"      : "Kauf",
        "Kurs"     : round(priceFirst, 2),
        "Anteile"  : round(shares, 4),
        "Wert"     : round(targetValue, 2),
        "CashFlow" : round(cashFlow, 2)
    })

    # --- Kursdaten durchgehen ---
    for datum, row in hist.iterrows():

        # Dividenden prüfen
        if str(datum.date()) in divsDates:
            idxDiv = 0
            while divsDates[idxDiv] != str(datum.date()): idxDiv += 1
            idxDiv = divs.keys()[idxDiv]
            divPerShare  = divs[idxDiv]
            divTotal     = shares * divPerShare
            cashFlow    += divTotal
            trades.append({
                "Datum"    : datum.date(),
                "Typ"      : "Dividende",
                "Kurs"     : round(divPerShare, 4),
                "Anteile"  : round(shares, 4),
                "Wert"     : round(divTotal, 2),
                "CashFlow" : round(cashFlow, 2)
            })

        # Positionswert prüfen
        posValue = shares * row["Close"]

        if posValue <= targetValue * (1 - thresBuy):
            if maxBuys is not None and cntBuys >= maxBuys:
                continue
            diffValue  = (targetValue - posValue) * tradeRatioBuy
            newShares  = diffValue / row["Close"]
            shares    += newShares
            cashFlow  -= diffValue
            cntBuys   += 1
            trades.append({
                "Datum"    : datum.date(),
                "Typ"      : "Nachkauf",
                "Kurs"     : round(row["Close"], 2),
                "Anteile"  : round(newShares, 4),
                "Wert"     : round(diffValue, 2),
                "CashFlow" : round(cashFlow, 2)
            })

        elif posValue >= targetValue * (1 + thresSell):
            diffValue  = (posValue - targetValue) * tradeRatioSell
            soldShares = diffValue / row["Close"]
            shares    -= soldShares
            cashFlow  += diffValue
            cntBuys    = 0
            trades.append({
                "Datum"    : datum.date(),
                "Typ"      : "Verkauf",
                "Kurs"     : round(row["Close"], 2),
                "Anteile"  : round(soldShares, 4),
                "Wert"     : round(diffValue, 2),
                "CashFlow" : round(cashFlow, 2)
            })

    # --- Abschluss: Position zum letzten Kurs auflösen ---
    priceLast  = hist["Close"].iloc[-1]
    finalValue = shares * priceLast
    cashFlow  += finalValue

    trades.append({
        "Datum"    : hist.index[-1].date(),
        "Typ"      : "Abschluss",
        "Kurs"     : round(priceLast, 2),
        "Anteile"  : round(shares, 4),
        "Wert"     : round(finalValue, 2),
        "CashFlow" : round(cashFlow, 2)
    })

    # --- Ausgabe ---
    tabTrades    = pd.DataFrame(trades)
    maxBuysStr   = str(maxBuys) if maxBuys is not None else "∞"

    print(f"Backtest: {symbol}  |  {start} → {end}")
    print(f"Zielwert: {targetValue}  |  Kaufschwelle: {thresBuy*100:.0f}%  |  Verkaufsschwelle: {thresSell*100:.0f}%")
    print(f"Kaufanteil: {tradeRatioBuy*100:.0f}%  |  Verkaufanteil: {tradeRatioSell*100:.0f}%  |  Max. Nachkäufe: {maxBuysStr}")
    print()
    print(tabTrades.to_string(index=False))
    print()
    print(f"Ergebnis:  {'+' if cashFlow >= 0 else ''}{cashFlow:.2f} €  ({'+' if cashFlow >= 0 else ''}{cashFlow/targetValue*100:.1f}%)")

    return tabTrades


# --- Aufruf ---
tabTrades = backtest(
    symbol         = "CRA1.F",
    start          = "2020-01-01",
    end            = "2025-12-31",
    targetValue    = 300,
    thresBuy       = 0.16,
    thresSell      = 0.20,
    tradeRatioBuy  = 0.50,
    tradeRatioSell = 1.00,
    maxBuys        = 2
)